This is the notebook that address the DOMAIN question requirements

A low-latency query returns the correct answer to a representative business question for the customer

In [0]:
%pip install -q databricks-sdk>=0.118.0 "psycopg[binary]>=3.1.0"
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Milestone 2.9 — Low-Latency Business Domain Query (Direct Lakebase Connection)
# Connects directly to Lakebase Postgres via psycopg2 using OAuth credentials
# from the Databricks SDK — no Spark/Lakehouse intermediary.

from databricks.sdk import WorkspaceClient
import psycopg
import pandas as pd


In [0]:
# Lakebase connection parameters ---
PROJECT = "meridian-bank"
BRANCH = "production"
ENDPOINT = "primary"
DATABASE = "databricks_postgres"
HOST = "ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net"

# Generate short-lived OAuth credential via Databricks SDK
w = WorkspaceClient()
username = w.current_user.me().user_name
token = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/{BRANCH}/endpoints/{ENDPOINT}"
).token


In [0]:
# Connect directly to Lakebase Postgres ---
conn = psycopg.connect(
    host=HOST,
    port=5432,
    dbname=DATABASE,
    user=username,
    password=token,
    sslmode="require",
)


In [0]:
# Business Query: High-value at-risk customers with NBA recommendations ---
query = """
SELECT
    cp.customer_id,
    cp.customer_display_name,
    cp.deposit_balance_usd,
    cp.tenure_years,
    cp.tier,
    ar.atrisk_product_id,
    ar.attrition_risk_score,
    ar.days_to_maturity,
    nba.recommended_action,
    nba.predicted_retained_usd,
    p.product_name AS recommended_product,
    p.rate_apy AS offer_rate
FROM meridian_bank.synced_gold_customer_position cp
JOIN meridian_bank.synced_gold_open_atrisk ar
    ON cp.customer_id = ar.customer_id
JOIN meridian_bank.synced_gold_nba_recommendations nba
    ON cp.customer_id = nba.customer_id
LEFT JOIN meridian_bank.products p
    ON nba.recommended_offer_product_id = p.product_id
WHERE cp.deposit_balance_usd > 100000
  AND ar.attrition_risk_score > 0.6
ORDER BY ar.attrition_risk_score DESC, cp.deposit_balance_usd DESC
LIMIT 10;
"""

with conn.cursor() as cur:
    cur.execute(query)
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

conn.close()

df = pd.DataFrame(rows, columns=columns)
print(f"✓ Returned {len(df)} high-risk customers from Lakebase (direct Postgres connection)")
print(df.to_string(index=False))


✓ Returned 10 high-risk customers from Lakebase (direct Postgres connection)
 customer_id customer_display_name  deposit_balance_usd  tenure_years     tier atrisk_product_id  attrition_risk_score  days_to_maturity recommended_action  predicted_retained_usd recommended_product offer_rate
CUST-0004138      Customer 0004138           1249678.51            19 affluent     PROD-DEP-2003                  0.95                26    retention_offer            77545.848390                None       None
CUST-0001326      Customer 0001326           1210046.73            13 affluent     PROD-DEP-2003                  0.95                 7    retention_offer            73513.893480                None       None
CUST-0005063      Customer 0005063           1209662.12             8 affluent     PROD-DEP-2001                  0.95                39    retention_offer            73677.233123                None       None
CUST-0006136      Customer 0006136           1178486.53            18 affluent 

In [0]:
%sql
-- Milestone 2.9b — Same business query via Lakebase synced tables in Unity Catalog
-- These synced_gold_* tables are the UC representation of Lakebase Postgres tables.
-- Data flows: UC Gold → create_synced_table → Lakebase Postgres → queryable here via UC.
-- Demonstrates governed access to Lakebase data through Spark SQL (UC permissions, lineage, auditing).

SELECT
    cp.customer_id,
    cp.customer_display_name,
    cp.deposit_balance_usd,
    cp.tenure_years,
    cp.tier,
    ar.atrisk_product_id,
    ar.attrition_risk_score,
    ar.days_to_maturity,
    nba.recommended_action,
    nba.predicted_retained_usd
FROM techsummit_27.meridian_bank.synced_gold_customer_position cp
JOIN techsummit_27.meridian_bank.synced_gold_open_atrisk ar
    ON cp.customer_id = ar.customer_id
JOIN techsummit_27.meridian_bank.synced_gold_nba_recommendations nba
    ON cp.customer_id = nba.customer_id
WHERE cp.deposit_balance_usd > 100000
  AND ar.attrition_risk_score > 0.6
ORDER BY ar.attrition_risk_score DESC, cp.deposit_balance_usd DESC
LIMIT 10

customer_id,customer_display_name,deposit_balance_usd,tenure_years,tier,atrisk_product_id,attrition_risk_score,days_to_maturity,recommended_action,predicted_retained_usd
CUST-0004138,Customer 0004138,1249678.51,19,affluent,PROD-DEP-2003,0.95,26,retention_offer,77545.84839000001
CUST-0001326,Customer 0001326,1210046.73,13,affluent,PROD-DEP-2003,0.95,7,retention_offer,73513.89348000001
CUST-0005063,Customer 0005063,1209662.12,8,affluent,PROD-DEP-2001,0.95,39,retention_offer,73677.23312250001
CUST-0006136,Customer 0006136,1178486.53,18,affluent,PROD-DEP-2001,0.95,16,retention_offer,73099.70874750002
CUST-0008393,Customer 0008393,1154432.3599999999,8,affluent,PROD-DEP-2002,0.95,29,retention_offer,71578.499895
CUST-0007616,Customer 0007616,1150936.5499999998,11,affluent,PROD-DEP-2002,0.95,18,retention_offer,71189.28737250001
CUST-0003546,Customer 0003546,1147772.9,17,affluent,PROD-DEP-2001,0.95,36,retention_offer,70971.58107000001
CUST-0001807,Customer 0001807,1130103.68,14,affluent,PROD-DEP-2003,0.95,33,retention_offer,68017.792425
CUST-0007246,Customer 0007246,1128826.8499999999,13,affluent,PROD-DEP-2001,0.95,15,retention_offer,68542.83320250001
CUST-0005692,Customer 0005692,1108446.13,19,private,PROD-DEP-2003,0.95,35,retention_offer,68587.880415


In [0]:
# Milestone 2.9c — Ask a business question in plain English via Genie
# Demonstrates the AI/BI Genie interface: a business user asks a question
# in natural language, and Genie translates it to SQL against the governed
# UC tables, returning structured results.
# this is a low latency business query

import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import GenieMessage

w = WorkspaceClient()

GENIE_SPACE_ID = "01f1a23899dc1441a01de338fa0482c7"  # Meridian Customer Retention

# Ask a business question in plain English
question = "Which customer tiers have the most balance at risk, and what's the total revenue at risk per tier?"

print(f"Asking Genie: \"{question}\"")
print("=" * 60)

# Start a conversation with the Genie space
conversation = w.genie.start_conversation(
    space_id=GENIE_SPACE_ID,
    content=question,
)

# Poll until the message is complete
msg = w.genie.get_message(
    space_id=GENIE_SPACE_ID,
    conversation_id=conversation.conversation_id,
    message_id=conversation.message_id,
)

# Poll until Genie finishes (COMPLETED or error)
TERMINAL = ("COMPLETED", "FAILED", "CANCELLED", "QUERY_RESULT_EXPIRED")
for _ in range(30):  # up to 60s
    if str(msg.status).split(".")[-1] in TERMINAL:
        break
    time.sleep(2)
    msg = w.genie.get_message(
        space_id=GENIE_SPACE_ID,
        conversation_id=conversation.conversation_id,
        message_id=conversation.message_id,
    )

print(f"Status: {msg.status}")

# Extract and display the query result
if msg.attachments:
    for attachment in msg.attachments:
        if attachment.query:
            print(f"\nGenerated SQL:\n  {attachment.query.query}")
            print(f"\nDescription: {attachment.query.description or 'N/A'}")

            # Fetch the result set
            try:
                result = w.genie.get_message_query_result(
                    space_id=GENIE_SPACE_ID,
                    conversation_id=conversation.conversation_id,
                    message_id=conversation.message_id,
                    attachment_id=attachment.id,
                )
                print(f"\nResults ({len(result.statement_response.result.data_array)} rows):")
                # Print column headers
                columns = [col.name for col in result.statement_response.manifest.schema.columns]
                print(f"  {' | '.join(columns)}")
                print(f"  {'-' * 60}")
                for row in result.statement_response.result.data_array:
                    print(f"  {' | '.join(str(v) for v in row)}")
            except Exception as e:
                print(f"  (Result fetch: {e})")
        if attachment.text:
            print(f"\nGenie says: {attachment.text.content}")
else:
    print("No attachments in response.")

print("\n" + "=" * 60)
print("✓ Business question answered via natural language — no SQL knowledge required.")

Asking Genie: "Which customer tiers have the most balance at risk, and what's the total revenue at risk per tier?"
Status: MessageStatus.COMPLETED

Generated SQL:
  WITH `tier_risk` AS (
  SELECT
    `tier`,
    MEASURE(`balance_at_risk`) AS `balance_at_risk`,
    MEASURE(`revenue_at_risk`) AS `revenue_at_risk`,
    RANK() OVER (ORDER BY MEASURE(`balance_at_risk`) DESC) AS `balance_at_risk_rank`
  FROM `techsummit_27`.`meridian_bank`.`mv_customer_risk`
  WHERE `tier` IS NOT NULL
  GROUP BY ALL
)
SELECT
  `tier`,
  `balance_at_risk`,
  `revenue_at_risk`
FROM `tier_risk`
ORDER BY `balance_at_risk_rank` ASC, `tier` ASC;

Description: You want to see the total balance at risk and total revenue at risk for each customer tier, ranked by the balance at risk from highest to lowest.
  (Result fetch: 'GenieAttachment' object has no attribute 'id')

Genie says: The **affluent** tier has the most balance at risk at **109,804,706.26**, with **2,826,517.66** in revenue at risk, followed by **private